# MAUDE Data Profiling & Visualization

FDA MAUDE device adverse-event lakehouse (`MAUDE_DB`) - full history, ~25.37M MDRs.

Container-runtime Workspace notebook (no Streamlit). Charts use **Plotly** (interactive)
and **matplotlib** (preinstalled). Plotly is installed from the Snowflake-managed PyPI
repo - run the install cell first.

**Governance:** MAUDE is passive surveillance. Counts cannot establish event rates or
causation and must not drive individual patient-care decisions.

## Install packages (Snowflake-managed PyPI repo)
The Snowflake PyPI artifact repository is available in all accounts by default (PUBLIC
role), so `!pip install` works with no admin setup. matplotlib ships in the runtime.

In [ ]:
# Installs from the default Snowflake-managed PyPI artifact repository.
!pip install plotly -q

In [ ]:
USE ROLE MAUDE_ENGINEER;
USE WAREHOUSE MAUDE_WH;
USE SCHEMA MAUDE_DB.CURATED;

## Load status (live)

In [ ]:
SELECT status, COUNT(*) AS partitions, SUM(loaded_records) AS rows_loaded
FROM MAUDE_DB.RAW.LOAD_CONTROL GROUP BY status ORDER BY status;

In [ ]:
SELECT
  (SELECT COUNT(*) FROM MAUDE_DB.RAW.RAW_DEVICE_EVENT)            AS raw_rows,
  (SELECT COUNT(*) FROM MAUDE_DB.CURATED.FACT_ADVERSE_EVENT)      AS fact_rows,
  (SELECT MAX(total_records) FROM MAUDE_DB.RAW.MANIFEST_HISTORY)  AS manifest_total;

## Load aggregates into pandas
Server-side `GROUP BY` keeps result sets tiny; we never pull raw rows.

In [ ]:
import pandas as pd
import plotly.express as px
import matplotlib.pyplot as plt
from snowflake.snowpark.context import get_active_session
session = get_active_session()

def q(sql):
    return session.sql(sql).to_pandas()

evt = q("""SELECT COALESCE(NULLIF(event_type,''),'(blank)') AS EVENT_TYPE, COUNT(*) AS N
            FROM MAUDE_DB.CURATED.FACT_ADVERSE_EVENT GROUP BY 1 ORDER BY N DESC""")
yr = q("""SELECT report_year AS REPORT_YEAR, COUNT(*) AS N
           FROM MAUDE_DB.CURATED.FACT_ADVERSE_EVENT WHERE report_year BETWEEN 1995 AND 2026
           GROUP BY 1 ORDER BY 1""")
evt_yr = q("""SELECT report_year AS REPORT_YEAR,
                 COALESCE(NULLIF(event_type,''),'(blank)') AS EVENT_TYPE, COUNT(*) AS N
              FROM MAUDE_DB.CURATED.FACT_ADVERSE_EVENT
              WHERE report_year BETWEEN 2010 AND 2026 GROUP BY 1,2 ORDER BY 1""")
spec = q("""SELECT medical_specialty AS SPECIALTY, COUNT(*) AS N
             FROM MAUDE_DB.CURATED.DIM_DEVICE
             WHERE medical_specialty IS NOT NULL AND medical_specialty <> '' AND medical_specialty <> 'Unknown'
             GROUP BY 1 ORDER BY N DESC LIMIT 15""")
narr = q("""SELECT text_type_code AS TEXT_TYPE, COUNT(*) AS SEGMENTS,
                ROUND(AVG(narrative_length)) AS AVG_LEN,
                ROUND(100*COUNT_IF(redaction_flag)/COUNT(*),2) AS PCT_REDACTED
             FROM MAUDE_DB.CURATED.EVENT_NARRATIVE GROUP BY 1 ORDER BY SEGMENTS DESC""")
print(f"Loaded aggregates - events:{len(evt)} years:{len(yr)} specialties:{len(spec)}")

## 1. MDRs by event type (Plotly)

In [ ]:
fig = px.bar(evt.sort_values('N'), x='N', y='EVENT_TYPE', orientation='h',
             title='MDRs by event type (full history)',
             labels={'N':'MDR count','EVENT_TYPE':'Event type'},
             color='EVENT_TYPE', text='N')
fig.update_layout(height=320, showlegend=False)
fig.show()

## 2. MDR volume by year (Plotly area)

In [ ]:
fig = px.area(yr, x='REPORT_YEAR', y='N', markers=True,
              title='MDR volume by year',
              labels={'N':'MDRs received','REPORT_YEAR':'Report year'})
fig.update_layout(height=320)
fig.show()

## 3. Top medical specialties (Plotly)

In [ ]:
fig = px.bar(spec.sort_values('N'), x='N', y='SPECIALTY', orientation='h',
             title='Top medical specialties by MDR volume',
             labels={'N':'MDR count','SPECIALTY':''},
             color='N', color_continuous_scale='Blues')
fig.update_layout(height=460, coloraxis_showscale=False)
fig.show()

## 4. Event-type mix by year (Plotly stacked area)

In [ ]:
fig = px.area(evt_yr, x='REPORT_YEAR', y='N', color='EVENT_TYPE',
              title='Event-type mix by year (2010-2026)',
              labels={'N':'MDRs','REPORT_YEAR':'Year','EVENT_TYPE':'Event type'})
fig.update_layout(height=420)
fig.show()

## 5. FOIA redaction rate by narrative type (matplotlib)

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(narr['TEXT_TYPE'], narr['PCT_REDACTED'], color=['#1f77b4', '#ff7f0e'])
ax.set_ylabel('% FOIA-redacted')
ax.set_title('FOIA redaction rate by narrative type')
ax.set_ylim(0, max(narr['PCT_REDACTED']) * 1.25)
for b, v in zip(bars, narr['PCT_REDACTED']):
    ax.text(b.get_x() + b.get_width()/2, v + 0.3, f'{v}%', ha='center')
plt.xticks(rotation=10)
plt.tight_layout()
plt.show()
print('(b)(4)=trade secret, (b)(6)=patient/personnel redactions flagged in EVENT_NARRATIVE')